# 🌲 California Air Quality Analysis (2015–2024)
## Notebook 01: Data Ingestion & Cleaning Pipeline

### 📌 Project Context
The Environmental Protection Agency (EPA) provides daily Air Quality Index (AQI) reports aggregated by county. Each year is released as an individual CSV file containing nationwide county records.

### 🎯 Objectives in this Notebook:
1. **Discover & Merge**: Scan `data/raw/` and iteratively concatenate 10 years of EPA daily files (2015 to 2024).
2. **State Filtering**: Retain solely records belonging to **California** (`State == 'California'`), discarding out-of-state records.
3. **Column Standardization**: Normalize column headers to lowercase snake_case (`county`, `date`, `aqi`, `category`, `defining_parameter`).
4. **Date Parsing & Formatting**: Standardize disparate date strings to strict ISO-8601 (`YYYY-MM-DD`).
5. **Data Quality & Null Handling**: Audit and address missing values, invalid numbers, and corrupted records.
6. **Deduplication**: Resolve duplicate county-date observations.
7. **Export**: Save the canonical clean dataset to `data/processed/ca_aqi_clean.csv`.

In [1]:
import os
import glob
import pandas as pd
import numpy as np

# Set project paths
BASE_DIR = os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd()
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
OUTPUT_PATH = os.path.join(PROCESSED_DIR, 'ca_aqi_clean.csv')

print(f"Raw data path:       {RAW_DIR}")
print(f"Processed data path: {OUTPUT_PATH}")

### Step 1: Discover and Inspect Raw CSV Files
We locate all yearly files in `data/raw/` matching `*.csv`.

In [2]:
raw_files = sorted(glob.glob(os.path.join(RAW_DIR, '*.csv')))
print(f"Found {len(raw_files)} raw yearly CSV files:")
for f in raw_files:
    print(f"  • {os.path.basename(f)} ({os.path.getsize(f) / 1024:.1f} KB)")

### Step 2: Merge Yearly Files & Standardize Headers
We map EPA schema variations to standardized column names.

In [3]:
dfs = []
for f in raw_files:
    df_year = pd.read_csv(f, low_memory=False)
    dfs.append(df_year)

raw_df = pd.concat(dfs, ignore_index=True)
print(f"Total raw rows loaded: {len(raw_df):,}")
print("\nRaw columns detected:", raw_df.columns.tolist())
raw_df.head(3)

### Step 3: Normalize Column Names & Filter for California

In [4]:
# Normalize column names
rename_map = {}
for col in raw_df.columns:
    clean_name = col.strip().lower().replace(' ', '_').replace('-', '_')
    if clean_name in ['state_name', 'state']:
        rename_map[col] = 'state'
    elif clean_name in ['county_name', 'county']:
        rename_map[col] = 'county'
    elif clean_name == 'date':
        rename_map[col] = 'date'
    elif clean_name == 'aqi':
        rename_map[col] = 'aqi'
    elif clean_name == 'category':
        rename_map[col] = 'category'
    elif clean_name in ['defining_parameter', 'defining_param', 'parameter']:
        rename_map[col] = 'defining_parameter'

raw_df.rename(columns=rename_map, inplace=True)

# Filter for California
if 'state' in raw_df.columns:
    state_mask = raw_df['state'].astype(str).str.strip().str.title().isin(['California', 'Ca'])
    non_ca = (~state_mask).sum()
    ca_df = raw_df[state_mask].copy()
    print(f"Filtered California records: {len(ca_df):,} (Dropped {non_ca:,} out-of-state records)")
else:
    ca_df = raw_df.copy()

# Clean county names: remove trailing ' County' and strip spaces
ca_df['county'] = ca_df['county'].astype(str).str.strip().str.replace(r'\s+County$', '', regex=True).str.title()
ca_df.head(3)

### Step 4: Parse Dates & Validate Numerical AQI Values

In [5]:
# Convert date to datetime and format as ISO YYYY-MM-DD
ca_df['date'] = pd.to_datetime(ca_df['date'], errors='coerce')
initial_len = len(ca_df)
ca_df = ca_df.dropna(subset=['date'])
ca_df['date'] = ca_df['date'].dt.strftime('%Y-%m-%d')

# Validate and convert AQI to integer
ca_df['aqi'] = pd.to_numeric(ca_df['aqi'], errors='coerce')
ca_df = ca_df.dropna(subset=['aqi'])
ca_df['aqi'] = ca_df['aqi'].astype(int)

print(f"Records retained after date and AQI validation: {len(ca_df):,} (Discarded {initial_len - len(ca_df)} invalid)")

### Step 5: Handle Missing Values & Categorize Imputations

In [6]:
def impute_category(aqi_val):
    if aqi_val <= 50: return 'Good'
    if aqi_val <= 100: return 'Moderate'
    if aqi_val <= 150: return 'Unhealthy for Sensitive Groups'
    if aqi_val <= 200: return 'Unhealthy'
    if aqi_val <= 300: return 'Very Unhealthy'
    return 'Hazardous'

# Check for null category
ca_df['category'] = ca_df['category'].fillna('').astype(str).str.strip()
missing_cat = (ca_df['category'] == '') | (ca_df['category'].str.lower() == 'nan')
if missing_cat.sum() > 0:
    print(f"Imputing {missing_cat.sum():,} missing categories using EPA breakpoints.")
    ca_df.loc[missing_cat, 'category'] = ca_df.loc[missing_cat, 'aqi'].apply(impute_category)

# Clean defining parameter
ca_df['defining_parameter'] = ca_df['defining_parameter'].fillna('PM2.5').astype(str).str.strip()
ca_df['defining_parameter'] = ca_df['defining_parameter'].replace({'': 'PM2.5', 'nan': 'PM2.5'})

print("Missing value summary after imputation:")
print(ca_df[['county', 'date', 'aqi', 'category', 'defining_parameter']].isna().sum())

### Step 6: Deduplication & Final Export to `data/processed/ca_aqi_clean.csv`

In [7]:
# Deduplicate on (county, date)
before_dedup = len(ca_df)
ca_df = ca_df.sort_values(by=['county', 'date', 'aqi'], ascending=[True, True, False])
ca_df = ca_df.drop_duplicates(subset=['county', 'date'], keep='first')

target_cols = ['county', 'date', 'aqi', 'category', 'defining_parameter']
clean_df = ca_df[target_cols].sort_values(by=['date', 'county']).reset_index(drop=True)

os.makedirs(PROCESSED_DIR, exist_ok=True)
clean_df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Successfully exported clean dataset: {OUTPUT_PATH}")
print(f"📊 Total clean observations: {len(clean_df):,}")
print(f"📅 Date span: {clean_df['date'].min()} to {clean_df['date'].max()}")
print(f"📍 Number of counties: {clean_df['county'].nunique()}")
clean_df.head(5)